# Analyze repository structure

> Can you help me analyze the structure of this repo?

<details><summary>8 previous messages</summary>

> I’ll take a first pass through the repo layout and key metadata, then I’ll summarize the architecture and likely entry points. I’m going to start with fast file discovery and a few top-level reads.
>
> <details><summary>Ran 3 commands</summary>
>
> - Ran `rg --files`
> - Ran `Get-ChildItem -Force`
> - Ran `git status --short`
>
> </details>
>
> The repo is compact and Python-only from the file listing. Git refused status because Windows reports a different owner for the checkout; I’ll leave that alone and continue with read-only structure analysis.
>
> <details><summary>Explored 2 files, ran a command</summary>
>
> <details><summary>Explored 2 files</summary>
>
> - Read `./config.py`
> - Read `./README.md`
>
> </details>
>
> - Ran `rg "^(class|def|if __name__|import|from )" -n`
>
> </details>
>
> The README is unusually detailed, though it appears to have encoding mojibake for symbols in this terminal view. I’m going one layer deeper now: imports, public functions, and how the experiment scripts call the core modules.
>
> <details><summary>Explored a file, ran 2 commands</summary>
>
> - Ran `Get-ChildItem -Recurse -Filter *.py | Select-Object FullName,Length`
>
> <details><summary>Explored a file</summary>
>
> - Read `./localization/pipeline.py`
>
> </details>
>
> - Ran `rg -n "^def |^class |^if __name__|^import |^from " localization signal_model transforms experiments utils config.py`
>
> </details>
>
> A few important contours are showing up: `localization/pipeline.py` is the real application API, while most scripts in `experiments/` are calibration or exploratory runs. I’m checking package boundaries and dependencies next, especially because the transforms use GPU libraries.
>
> <details><summary>Explored 4 files</summary>
>
> - Read `./localization/roi.py`
> - Read `./transforms/prt.py`
> - Read `./transforms/hrt.py`
> - Read `./signal_model/propagation.py`
>
> </details>
</details>

Here’s the repo shape in plain terms.

**What This Repo Is**

This is a compact Python research prototype for near-field localization using a two-stage signal-processing pipeline:

```text
simulate received ULA signal
-> add noise
-> matched filter
-> PRT coarse localization
-> build ROI
-> HRT refinement
-> final polar/cartesian position
```

The main production-ish API is [localization/pipeline.py](./localization/pipeline.py), especially:

```python
localize(rx, time_axis, antenna_x, theta_hint=None, tau_hint=None)
```

**Top-Level Layout**

```text
config.py              Shared constants and system parameters
README.md              Detailed project explanation, but symbol encoding looks corrupted in terminal

signal_model/          Signal generation and preprocessing
transforms/            GPU PRT and HRT implementations
localization/          Mapping, ROI construction, full localization pipeline
experiments/           Calibration, validation, Monte Carlo, SIC/DBSCAN experiments
utils/                 Plotting helpers
```

**Core Modules**

[config.py](./config.py) defines the physical setup:

- carrier frequency, bandwidth, sampling rate
- array geometry
- simulation noise/SNR settings
- PRT discretization

[signal_model/](./signal_model) contains the simulator:

- `gaussian_source.py`: Gaussian pulse
- `propagation.py`: exact spherical-wave propagation and received signal synthesis
- `noise.py`: AWGN
- `matched_filter.py`: matched filtering per antenna channel

[transforms/](./transforms) contains the expensive math:

- [transforms/prt.py](./transforms/prt.py): GPU-accelerated Parabolic Radon Transform
- [transforms/hrt.py](./transforms/hrt.py): GPU-accelerated Hyperbolic Radon Transform

Both require `cupy` and a CUDA-capable NVIDIA GPU.

[localization/](./localization) is the main orchestration layer:

- `parameter_mapping.py`: converts between physical `(R, theta)` and PRT `(p, q)`
- `roi.py`: builds the HRT search region around the PRT estimate
- `pipeline.py`: complete `matched_filter -> PRT -> ROI -> HRT` pipeline

**Experiments**

[experiments/](./experiments) is not just one experiment; it is a development history of the method:

- `exp01_prt_range_error.py`: PRT range error study
- `q_reso.py`: q-grid resolution study
- `find_roi.py`: ROI calibration
- `roi_test.py`: ROI coverage validation
- `hrt_tau_reso.py`: HRT tau resolution study
- `hrt_test.py`: standalone HRT validation
- `single_user.py`: main end-to-end Monte Carlo experiment
- `SIC*.py`, `DBSCAN*.py`, `test_SIC.py`: multi-user / clustering / cancellation experiments

The most important user-facing experiment appears to be [experiments/single_user.py](./experiments/single_user.py).

**Notable Structural Observations**

There are no `__init__.py` files, so this is being run as a loose script-style repo rather than a formal Python package.

There is no `requirements.txt`, `pyproject.toml`, or environment file. The README lists dependencies manually: `numpy`, `scipy`, `matplotlib`, `tqdm`, and the code also uses `cupy` and `sklearn` in some experiments.

The README is detailed but the terminal output shows encoding corruption for symbols like arrows, theta, tau, degrees, etc. The source file may be UTF-8 read through the wrong code page, or it may already contain mojibake.

Git status could not run because the repo is marked as “dubious ownership” by Git on Windows. I did not change global Git config.

**Mental Model**

The clean dependency direction is roughly:

```text
config
  ↓
signal_model
  ↓
transforms
  ↓
localization
  ↓
experiments
```

The one to read first is `localization/pipeline.py`; the one to run first is probably `experiments/single_user.py`, assuming CUDA/CuPy is available.


### System Env Activation 

H:\virtual_envs\prt_exp\Scripts\activate.bat